<a href="https://colab.research.google.com/github/Ciano-Domenico/Progetto_EVWSD_ML/blob/Domenico/progetto_EVWSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Drive e setup cartelle
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Progetto_EVWSD_ML"
IMAGES_DIR = os.path.join(DRIVE_PROJECT_PATH, "data/images")
HF_CACHE_DIR = os.path.join(DRIVE_PROJECT_PATH, "data/hf_cache")
IMAGES_ROOT  = os.path.join(IMAGES_DIR, "imgs")

os.makedirs(HF_CACHE_DIR, exist_ok=True)


In [ ]:
!pip install -q sentence-transformers datasets pillow torch

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import zipfile

dataset = load_dataset("swap-uniba/EVWSD-ITA", cache_dir=HF_CACHE_DIR)

# scarica immagini solo se non già presenti
if not os.path.exists(IMAGES_ROOT):
    zip_path = hf_hub_download(
        repo_id="swap-uniba/EVWSD-ITA",
        filename="imgs.zip",
        repo_type="dataset",
        cache_dir=HF_CACHE_DIR
    )
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(IMAGES_DIR)
    print("Immagini estratte.")
else:
    print("Immagini già presenti.")

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# vision encoder: CLIP base (stesso backbone del modello multilingue)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("Modello CLIP caricato.")

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1", device=device)
text_model.eval()
print("Modello SentenceTransformer caricato.")

In [ ]:
import random
from PIL import Image

IMG_EXT = ".png"
N_CANDIDATE = 10

def load_image(img_path_str):
    full_path = os.path.join(IMAGES_ROOT, img_path_str + IMG_EXT)
    if not os.path.exists(full_path):
        return None
    return Image.open(full_path).convert("RGB")

def encode_images(pil_images, model, processor, device):
    inputs = processor(images=pil_images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        vision_out = model.vision_model(pixel_values=inputs["pixel_values"])
        emb = model.visual_projection(vision_out.pooler_output)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu()

def scegli_pacchetto_casuale(dataset, n=N_CANDIDATE):
    idx = random.randint(0, len(dataset["train"]) - 1)
    row = dataset["train"][idx]

    # 1. prendi co-iponimi (più simili)
    # 2. poi stesso lemma senso diverso
    # 3. se non bastano, riempi con immagini casuali da altre istanze
    candidati = {}  # path → img, mantiene unicità

    for path, is_co in zip(row["images"], row["is_co_hyp"]):
        if path not in candidati:
            img = load_image(path)
            if img is not None:
                candidati[path] = (img, "co_hyp" if is_co else "same_lemma")

    # se non bastano riempi con immagini casuali da altre istanze
    tentativi = 0
    while len(candidati) < n and tentativi < 1000:
        tentativi += 1
        row_rand = dataset["train"][random.randint(0, len(dataset["train"]) - 1)]
        for path in row_rand["images"]:
            if path not in candidati and len(candidati) < n:
                img = load_image(path)
                if img is not None:
                    candidati[path] = (img, "random")

    # se ci sono più di n, priorità: co_hyp > same_lemma > random
    if len(candidati) > n:
        co_hyp_paths   = [p for p, (_, t) in candidati.items() if t == "co_hyp"]
        same_lem_paths = [p for p, (_, t) in candidati.items() if t == "same_lemma"]

        paths = co_hyp_paths + same_lem_paths
        if len(paths) < n:
            random_paths = [p for p, (_, t) in candidati.items() if t == "random"]
            paths += random.sample(random_paths, n - len(paths))
        elif len(paths) > n:
            paths = paths[:n]
    else:
        paths = list(candidati.keys())

    random.shuffle(paths)
    immagini = [candidati[p][0] for p in paths]
    tipi     = [candidati[p][1] for p in paths]

    embeddings_candidati = encode_images(immagini, clip_model, clip_processor, device)

    metadati = {
        "id":           row["id"],
        "lemma":        row["lemma"],
        "gloss":        row["gloss"],
        "n_co_hyp":     tipi.count("co_hyp"),
        "n_same_lemma": tipi.count("same_lemma"),
        "n_random":     tipi.count("random"),
        "n_candidate":  len(immagini),
    }

    return immagini, embeddings_candidati, metadati

In [ ]:
import matplotlib.pyplot as plt

immagini, embeddings_candidati, metadati = scegli_pacchetto_casuale(dataset)

print(f"Istanza:         {metadati['id']}")
print(f"Lemma:           {metadati['lemma']}")
print(f"Gloss:           {metadati['gloss']}")
print(f"Co-iponimi:      {metadati['n_co_hyp']}")
print(f"Stesso lemma:    {metadati['n_same_lemma']}")
print(f"Casuali:         {metadati['n_random']}")
print(f"Totale:          {metadati['n_candidate']}")
print(f"Shape embeddings: {embeddings_candidati.shape}")

fig, axes = plt.subplots(1, len(immagini), figsize=(3 * len(immagini), 3))
for i, (ax, img) in enumerate(zip(axes, immagini)):
    ax.imshow(img)
    ax.set_title(f"Candidata {i}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Ricerca immagini per testo

In [ ]:
# Funzione per calcolare la similarità del coseno
def cosine_similarity(embedding1, embedding2):
    return torch.nn.functional.cosine_similarity(embedding1, embedding2, dim=-1)

# Prendi l'input della frase dall'utente
user_query = input("Inserisci una frase per cercare le immagini: ")

prompt_sistema_clip = (
    "Traduci ed espandi la seguente richiesta di ricerca in una descrizione visiva letterale e oggettiva in inglese. "
    "Dettaglia solo elementi concreti: soggetti principali, oggetti in primo piano e sullo sfondo, colori dominanti e "
    "stile (es. fotografia, rendering 3D). Evita metafore, concetti astratti o opinioni. Restituisci solo la descrizione."
)

prompt_completo_clip = f"{prompt_sistema_clip}\n\nRichiesta: {user_query}"

# Chiede a Gemini di espandere la richiesta per CLIP
sentence_input = chiedi_a_gemini(prompt_completo_clip)
print(f"Descrizione espansa per CLIP: {sentence_input}")

# Codifica la frase (ora la descrizione espansa da Gemini)
with torch.no_grad():
    text_embedding = text_model.encode(sentence_input, convert_to_tensor=True, show_progress_bar=False)
    text_embedding = text_embedding.unsqueeze(0).to(device)  # Aggiungi una dimensione per il batch e sposta al device
    text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True) # Normalizza l'embedding del testo

# Calcola la similarità del coseno con tutti gli embeddings delle immagini
similarities = cosine_similarity(text_embedding, embeddings_candidati.to(device))

# Ordina le immagini in base alla similarità
sorted_indices = torch.argsort(similarities, descending=True)
sorted_similarities = similarities[sorted_indices]
sorted_images = [immagini[i] for i in sorted_indices]

# Visualizza le immagini ordinate e i loro punteggi di similarità
print("Immagini ordinate per similarità con la frase:\n")
fig, axes = plt.subplots(1, len(sorted_images), figsize=(3 * len(sorted_images), 3))
if len(sorted_images) == 1:
    axes = [axes] # Ensure axes is iterable even for a single image

for i, (ax, img) in enumerate(zip(axes, sorted_images)):
    ax.imshow(img)
    ax.set_title(f"Sim: {sorted_similarities[i].item():.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q transformers accelerate requests google-genai

In [ ]:
import os
from google import genai

# Sostituisci con la tua chiave ottenuta da Google AI Studio
GEMINI_API_KEY = "AQ.Ab8RN6Kma1ZVsFCRAXFsR1eFUNewCIoDwWLvdm3OqCn_M7K8Cg"
client_gemini = genai.Client(api_key=GEMINI_API_KEY)

def chiedi_a_gemini(prompt_testuale):
  """
  Funzione generica che invia un prompt testuale a Gemini e restituisce la risposta pulita.
  """
  try:
      response = client_gemini.models.generate_content(
          model='gemini-2.5-flash-lite',
          contents=prompt_testuale,
      )
      # Restituisce il testo senza spazi vuoti o virgolette esterne superflue
      return response.text.strip().replace('"', '')
  except Exception as e:
      print(f"Errore durante la chiamata a Gemini: {e}")
      return None

In [ ]:
import torch
import requests
from io import BytesIO
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

print("Fase 1: Caricamento del processore e del modello nella GPU...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("Modello caricato con successo!")

In [ ]:
def calcola_score_vqa_unitor(image_pil, question):
    try:
        immagine = image_pil

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": "Image"},
                    {"type": "text", "text": question}
                ]
            }
        ]

        testo_formattato = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[testo_formattato], images=immagine, padding=True, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            next_token_logits = outputs.logits[0, -1, :]

        id_yes = processor.tokenizer.convert_tokens_to_ids("Yes")
        id_no = processor.tokenizer.convert_tokens_to_ids("No")

        logit_yes = next_token_logits[id_yes].float().item()
        logit_no = next_token_logits[id_no].float().item()

        logits_selezionati = torch.tensor([logit_yes, logit_no])
        probabilita = torch.softmax(logits_selezionati, dim=0)

        return probabilita[0].item()

    except Exception as e:
        print(f"Errore durante l'elaborazione: {e}")
        return None

In [ ]:
prompt_sistema = (
    "Trasforma la seguente richiesta dell'utente in una singola domanda visiva (Sì/No) "
    "in lingua inglese per un modello VQA. La domanda deve essere oggettiva, focalizzata solo "
    "su elementi concreti e visibili come soggetti, oggetti, colori o posizioni, escludendo "
    "pareri astratti o estetici. Restituisci esclusivamente il testo della domanda, senza introduzioni o spiegazioni."
)

# Uniamo i testi in un unico blocco prima di passarlo alla funzione
prompt_completo = f"{prompt_sistema}\n\nRichiesta dell'utente: {sentence_input}"

domanda_inglese = chiedi_a_gemini(prompt_completo)
print(f"Domanda generata: {domanda_inglese}")

vqa_results = []
print("\nFase 2: Calcolo dello score VQA per ciascuna immagine...")

# `immagini` is a list of PIL.Image.Image objects from `scegli_pacchetto_casuale`
for i, img in enumerate(immagini):
    # Call the modified function which now accepts a PIL Image directly
    score = calcola_score_vqa_unitor(img, domanda_inglese)
    if score is not None:
        vqa_results.append((score, img))
    print(f"  Processed image {i+1}/{len(immagini)} with score: {score:.4f}")

if vqa_results:
    # Sort the results by score in descending order
    vqa_results.sort(key=lambda x: x[0], reverse=True)

    sorted_scores = [res[0] for res in vqa_results]
    sorted_images = [res[1] for res in vqa_results]

    print("\nImmagini ordinate per score VQA con la domanda:\n")
    fig, axes = plt.subplots(1, len(sorted_images), figsize=(3 * len(sorted_images), 3))

    # Ensure axes is iterable even for a single image
    if len(sorted_images) == 1:
        axes = [axes]

    for i, (ax, img) in enumerate(zip(axes, sorted_images)):
        ax.imshow(img)
        ax.set_title(f"Score: {sorted_scores[i]:.2f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Nessun punteggio VQA è stato calcolato.")

### Combinazione dei punteggi CLIP e VQA

In [ ]:
# Prendi l'input della frase dall'utente (riutilizzando l'input precedente)
# sentence_input = input("Inserisci una frase per cercare le immagini: ")

print(f"Frase usata per la ricerca: {sentence_input}")

# --- Calcolo della similarità CLIP (come fatto in precedenza) ---
with torch.no_grad():
    text_embedding = text_model.encode(sentence_input, convert_to_tensor=True, show_progress_bar=False)
    text_embedding = text_embedding.unsqueeze(0).to(device)  # Aggiungi una dimensione per il batch e sposta al device
    text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True) # Normalizza l'embedding del testo

clip_similarities = cosine_similarity(text_embedding, embeddings_candidati.to(device)).cpu().numpy()

print("\nFase 3: Combinazione dei punteggi CLIP e VQA...")

combined_results = []
for i, img in enumerate(immagini):
    # Ottieni lo score VQA già calcolato (se disponibile)
    # Assumiamo che vqa_results sia ordinato in base al punteggio, ma l'ordine delle immagini
    # in `immagini` è quello originale. Dobbiamo recuperare il vqa_score per l'immagine corrente.
    # Per semplicità, ricalcolo lo score VQA qui per garantire la corrispondenza con l'immagine attuale.
    # In un'applicazione reale, sarebbe meglio salvare i risultati VQA in un dizionario o lista ordinata per l'indice dell'immagine.
    vqa_score = calcola_score_vqa_unitor(img, domanda_inglese)

    if vqa_score is not None:
        # Somma i punteggi CLIP e VQA
        # È possibile ponderare i punteggi se uno è considerato più importante
        combined_score = clip_similarities[i] + vqa_score
        combined_results.append((combined_score, img))

if combined_results:
    # Ordina i risultati combinati in ordine decrescente
    combined_results.sort(key=lambda x: x[0], reverse=True)

    sorted_combined_scores = [res[0] for res in combined_results]
    sorted_images_combined = [res[1] for res in combined_results]

    print("\nImmagini ordinate per score combinato (CLIP + VQA) con la domanda:\n")
    fig, axes = plt.subplots(1, len(sorted_images_combined), figsize=(3 * len(sorted_images_combined), 3))

    if len(sorted_images_combined) == 1:
        axes = [axes]

    for i, (ax, img) in enumerate(zip(axes, sorted_images_combined)):
        ax.imshow(img)
        ax.set_title(f"Score: {sorted_combined_scores[i]:.2f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Nessun punteggio combinato è stato calcolato.")